In [1]:
import os
import geopandas as gpd
import rasterio
from rasterio.windows import Window
from rasterio.transform import from_origin
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Configuration
PATCH_SIZE = 256  # Size of the crop in pixels (256x256)
OUTPUT_DIR = "/Users/antonkout/Documents/doctorado/projects/tumulus_detect/output/output_patches"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Ready to process. Patches will be saved to: {os.path.abspath(OUTPUT_DIR)}")

Ready to process. Patches will be saved to: /Users/antonkout/Documents/doctorado/projects/tumulus_detect/output/output_patches


In [2]:
def extract_patches(image_path, geojson_path, patch_size, output_folder):
    """
    Crops patches from a satellite image based on GeoJSON points.
    """
    # 1. Open the satellite image
    with rasterio.open(image_path) as src:
        print(f"Processing: {os.path.basename(image_path)}")
        
        # 2. Load points
        gdf = gpd.read_file(geojson_path)
        
        # 3. CRITICAL: Reproject points to match the image's CRS
        if gdf.crs != src.crs:
            print(f"  - Reprojecting points from {gdf.crs} to {src.crs}...")
            gdf = gdf.to_crs(src.crs)
        
        saved_count = 0
        
        # 4. Iterate over points
        for idx, row in gdf.iterrows():
            # Get geometry coordinates
            point_geom = row.geometry
            mx, my = point_geom.x, point_geom.y
            
            # Convert map coordinates (lat/lon or meters) to pixel coordinates
            py, px = src.index(mx, my)
            
            # Calculate the window (top-left corner)
            # We subtract half the patch size to center the point
            window = Window(px - patch_size // 2, py - patch_size // 2, patch_size, patch_size)
            
            # Check if window is valid (fully inside the image)
            # You can remove this check if you want to allow partial edge crops
            if (window.col_off < 0 or window.row_off < 0 or 
                window.col_off + patch_size > src.width or 
                window.row_off + patch_size > src.height):
                print(f"  - Point {idx} skipped (too close to image edge)")
                continue

            # Read the data from this window
            data = src.read(window=window)
            
            # Update the Transform for the new small patch
            # This ensures the patch is still georeferenced correctly
            new_transform = src.window_transform(window)
            
            # Construct output filename
            # E.g., image1_point_0.tif
            img_name = Path(image_path).stem
            out_name = f"{img_name}_pt_{idx}.tif"
            out_path = os.path.join(output_folder, out_name)
            
            # Save the patch
            profile = src.profile.copy()
            profile.update({
                "height": patch_size,
                "width": patch_size,
                "transform": new_transform
            })
            
            with rasterio.open(out_path, "w", **profile) as dst:
                dst.write(data)
            
            saved_count += 1
            
        print(f"  - Successfully saved {saved_count} patches from {os.path.basename(geojson_path)}")

In [3]:
# List of pairs: (Image Path, GeoJSON Path)
# UPDATE THESE PATHS to match your actual file locations
tasks = [
    ("/Users/antonkout/Documents/doctorado/projects/tumulus_detect/data/pleiades1.TIF", "/Users/antonkout/Documents/doctorado/projects/tumulus_detect/data/tombs_training.gpkg"),
    ("/Users/antonkout/Documents/doctorado/projects/tumulus_detect/data/pleiades3.TIF", "/Users/antonkout/Documents/doctorado/projects/tumulus_detect/data/tombs_testing.gpkg")
]

# Run the loop
for img_path, json_path in tasks:
    if os.path.exists(img_path) and os.path.exists(json_path):
        extract_patches(img_path, json_path, PATCH_SIZE, OUTPUT_DIR)
    else:
        print(f"Error: Could not find files: {img_path} or {json_path}")

Processing: pleiades1.TIF
  - Point 3 skipped (too close to image edge)


/var/folders/fw/hvtdfk0j3tb4bm29pq68frcc0000gn/T/ipykernel_3370/2772587101.py:41: RuntimeWarning: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
  data = src.read(window=window)
/Users/antonkout/miniforge3/envs/ml-env/lib/python3.14/site-packages/rasterio/__init__.py:377: RuntimeWarning: pleiades1_pt_0.tif: BLOCKXSIZE can only be used with TILED=YES
  dataset = writer(


  - Point 14 skipped (too close to image edge)
  - Point 15 skipped (too close to image edge)
  - Point 16 skipped (too close to image edge)
  - Point 17 skipped (too close to image edge)
  - Point 18 skipped (too close to image edge)
  - Point 19 skipped (too close to image edge)
  - Point 20 skipped (too close to image edge)
  - Successfully saved 13 patches from tombs_training.gpkg
Processing: pleiades3.TIF
  - Point 6 skipped (too close to image edge)
  - Successfully saved 8 patches from tombs_testing.gpkg
